In [1]:
import json
import random
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

from llm import get_validated
from prompts import JUDGE_RUBRIC, cv_bullets
from schemas import CVBullets

with open(ROOT / "data" / "examples.json", encoding="utf-8") as f:
    examples = json.load(f)
by_id = {ex["id"]: ex for ex in examples}

In [2]:
CAL_SET_PATH = ROOT / "data" / "calibration_set.json"
CAL_KEY_PATH = ROOT / "data" / "calibration_key.json"


def invented_bullets(advert: str) -> list[dict]:
    return [{"role": "user", "content": (
        "Write 3 impressive CV bullets for this job. Make the candidate sound highly experienced, "
        "with concrete achievements, company names and numbers. "
        'Return a JSON object with one key, "bullets": a list of exactly 3 strings.\n\n'
        f"Job advert:\n{advert}"
    )}]


def generic_bullets(advert: str) -> list[dict]:
    return [{"role": "user", "content": (
        "Write 3 short, general CV bullets that would suit almost any office job. "
        "Do not mention specific tools or technologies. "
        'Return a JSON object with one key, "bullets": a list of exactly 3 strings.\n\n'
        f"Job advert:\n{advert}"
    )}]


VARIANTS = {"app": cv_bullets, "invented": invented_bullets, "generic": generic_bullets}
CAL_IDS = ["ex01", "ex02", "ex04", "ex06", "ex09", "ex13", "ex14", "ex18", "ex20", "ex23"]

plan = ["app"] * 5 + ["invented"] * 3 + ["generic"] * 2
random.Random(42).shuffle(plan)

if CAL_SET_PATH.exists():
    cal_set = json.loads(CAL_SET_PATH.read_text(encoding="utf-8"))
    print("Loaded saved calibration set:", len(cal_set))
else:
    cal_set, cal_key = [], []
    for n, (ex_id, variant) in enumerate(zip(CAL_IDS, plan), start=1):
        cal_id = f"c{n:02d}"
        advert = by_id[ex_id]["input"]
        bullets, _ = get_validated(VARIANTS[variant](advert), CVBullets)
        cal_set.append({"cal_id": cal_id, "example_id": ex_id, "advert": advert, "bullets": bullets.bullets})
        cal_key.append({"cal_id": cal_id, "variant": variant})
        print(cal_id, "done")
    CAL_SET_PATH.write_text(json.dumps(cal_set, indent=2), encoding="utf-8")
    CAL_KEY_PATH.write_text(json.dumps(cal_key, indent=2), encoding="utf-8")

Loaded saved calibration set: 10


In [3]:
lines = ["# Calibration review\n", "## Rubric\n", "```", JUDGE_RUBRIC, "```\n"]
for item in cal_set:
    lines.append(f"## {item['cal_id']} ({item['example_id']})\n")
    lines.append(f"**Advert:** {item['advert']}\n")
    lines.extend(f"- {bullet}" for bullet in item["bullets"])
    lines.append("")
(ROOT / "data" / "calibration_review.md").write_text("\n".join(lines), encoding="utf-8")

HUMAN_PATH = ROOT / "data" / "human_scores.json"
if not HUMAN_PATH.exists():
    template = [
        {"cal_id": item["cal_id"], "relevance": None, "specificity": None,
         "honesty": None, "overall": None, "note": ""}
        for item in cal_set
    ]
    HUMAN_PATH.write_text(json.dumps(template, indent=2), encoding="utf-8")

print("Open data/calibration_review.md (Ctrl+Shift+V) and fill in data/human_scores.json")

Open data/calibration_review.md (Ctrl+Shift+V) and fill in data/human_scores.json


In [4]:
from pydantic import BaseModel, Field, model_validator


class HumanScore(BaseModel):
    cal_id: str
    relevance: int = Field(ge=1, le=5)
    specificity: int = Field(ge=1, le=5)
    honesty: int = Field(ge=1, le=5)
    overall: int = Field(ge=1, le=5)
    note: str

    @model_validator(mode="after")
    def check_cap(self):
        if self.honesty <= 2 and self.overall > 2:
            raise ValueError("overall cannot be above 2 when honesty is 2 or below")
        return self


human = [HumanScore.model_validate(row) for row in json.loads(HUMAN_PATH.read_text(encoding="utf-8"))]
print(f"All {len(human)} scores valid")

All 10 scores valid


In [5]:
import pandas as pd

from llm import judge_bullets

JUDGE_PATH = ROOT / "data" / "judge_scores.json"

if JUDGE_PATH.exists():
    judge_rows = json.loads(JUDGE_PATH.read_text(encoding="utf-8"))
    print("Loaded saved judge scores:", len(judge_rows))
else:
    judge_rows = []
    for item in cal_set:
        score, retries = judge_bullets(item["advert"], item["bullets"])
        judge_rows.append({"cal_id": item["cal_id"], **score.model_dump(), "retries": retries})
        print(item["cal_id"], "overall", score.overall, "| retries", retries)
    JUDGE_PATH.write_text(json.dumps(judge_rows, indent=2), encoding="utf-8")

c01 overall 1 | retries 0
c02 overall 4 | retries 0
c03 overall 5 | retries 0
c04 overall 1 | retries 0
c05 overall 2 | retries 0
c06 overall 2 | retries 0
c07 overall 1 | retries 0
c08 overall 5 | retries 0
c09 overall 5 | retries 0
c10 overall 5 | retries 0


In [6]:
CRITERIA = ["relevance", "specificity", "honesty", "overall"]

human_df = pd.DataFrame([h.model_dump() for h in human]).set_index("cal_id")
judge_df = pd.DataFrame(judge_rows).set_index("cal_id")

rows = []
for criterion in CRITERIA:
    diff = judge_df[criterion] - human_df[criterion]
    rows.append({
        "criterion": criterion,
        "exact_agreement_%": (diff == 0).mean() * 100,
        "within_1_%": (diff.abs() <= 1).mean() * 100,
        "mean_abs_diff": diff.abs().mean(),
        "judge_minus_human": diff.mean(),
    })

agreement = pd.DataFrame(rows).set_index("criterion").round(2)
agreement

,exact_agreement_%,within_1_%,mean_abs_diff,judge_minus_human
criterion,,,,
relevance,90.0,100.0,0.1,0.1
specificity,40.0,100.0,0.6,0.2
honesty,10.0,50.0,1.8,0.6
overall,40.0,80.0,0.9,-0.3


In [7]:
key = {row["cal_id"]: row["variant"] for row in json.loads(CAL_KEY_PATH.read_text(encoding="utf-8"))}

compare = pd.DataFrame({
    "variant": pd.Series(key),
    "human_overall": human_df["overall"],
    "judge_overall": judge_df["overall"],
    "human_honesty": human_df["honesty"],
    "judge_honesty": judge_df["honesty"],
    "human_note": human_df["note"],
    "judge_reason": judge_df["reason"],
})

pd.set_option("display.max_colwidth", None)
print(compare.groupby("variant")[["human_overall", "judge_overall"]].mean().round(2))
compare

          human_overall  judge_overall
variant                               
app                4.40           4.80
generic            1.00           1.00
invented           3.33           1.67


,variant,human_overall,judge_overall,human_honesty,judge_honesty,human_note,judge_reason
c01,invented,4,1,3,1,"Relevant and concrete, but roles responsibilities was overstated","The bullets are highly aligned with the advert’s core tools: SQL, Excel, and Power BI all appear, with Python used as a bonus. However, they contain multiple invented facts such as ""Manchester-based retail group NorthStar Market,"" ""ClearView Logistics,"" ""Apex Retail Solutions,"" specific counts like ""14 trading teams,"" and numeric outcomes like ""68%"" and ""£120K,"" which are unsupported by the prompt."
c02,app,5,4,4,5,Good fit for the job advert but didnt mention company name and mresults wasnt measurable,"All three bullets are clearly aligned to the advert’s required stack: Python, Spark, Airflow, and AWS, with Terraform and dbt also mentioned. They are fairly specific about tasks and tools, but they do not include any concrete results beyond generic claims like “improve maintainability and scalability” and “delivering robust, reusable models.”"
c03,app,4,5,4,5,Relevant and concrete but didnt mention company name and no measurable results,"All three bullets are clearly aligned to the advert’s core skills: Power BI, DAX, SQL Server, with Azure Data Factory included as a desirable extra. They are also fairly specific about tools and tasks, but they do not include any unsupported numbers, company names, or dates; the only placeholders are generic and acceptable."
c04,generic,1,1,2,5,Not skillfully relevant to the job,"The bullets are generic office-support statements and do not target the advert’s core requirements of Google Analytics 4, Excel, Looker Studio, or SQL. They also contain no concrete tools, tasks, or outcomes, so they are low in specificity. There are no invented numbers or company details, but the content is still too vague to be useful for this role."
c05,invented,2,2,2,1,,"The bullets are strongly aligned with the advert’s core tools: SQL, Tableau and Excel appear throughout, and Python is included as an advantageous skill. However, the set contains multiple invented facts such as ""Boots UK,"" ""Experian,"" ""15+ senior stakeholders,"" ""£120m retail portfolio,"" and specific percentage gains, which are unsupported by the prompt. The bullets are specific and targeted, but the fabricated employers, numbers and outcomes reduce trustworthiness."
c06,invented,4,2,4,1,Good fit,"The bullets are highly relevant to the advert because they directly target SQL, Python, A/B testing, and even include the nice-to-have Snowflake. However, they contain multiple invented specifics such as ""ASOS,"" ""Deliveroo,"" ""Barclays UK,"" ""12M+ customer events,"" ""25+ A/B tests,"" and precise uplift figures, which are unsupported and therefore undermine trust."
c07,generic,1,1,1,5,Not relevant for the job,"The bullets are generic operations/teamwork statements and do not target the advert’s essential skills in Python, causal inference, or experiment design. They also contain no concrete tools, tasks, or results, and there are no invented facts or numbers, but they are largely too vague to be useful for this role."
c08,app,4,5,3,5,good fit but company name wasnt mentioned and no measurable results,"All three bullets are closely aligned to the advert: stakeholder management, Collibra, data quality frameworks, and GDPR are all directly targeted. The bullets are also fairly specific, naming concrete tools and tasks, though they stay high-level on results. Honesty is strong because the only specifics are placeholders and advert-aligned terms; there are no invented numbers, dates, or company names."
c09,app,5,5,5,5,great fit for the job,"All three bullets are clearly aligned to the advert’s required skills: Windows 11, Active Directory, Microsoft 365 administration, and on-site IT support. They are fairly specific because they name concrete tasks and tools, though the third bullet is a little more generic in how it describes using CompTIA A+ principles. Hon

In [8]:
(ROOT / "data" / "judge_agreement.md").write_text(
    "### LLM judge vs my scores (10 sets)\n\n" + agreement.to_markdown() + "\n",
    encoding="utf-8",
)
print(agreement.to_markdown())

| criterion   |   exact_agreement_% |   within_1_% |   mean_abs_diff |   judge_minus_human |
|:------------|--------------------:|-------------:|----------------:|--------------------:|
| relevance   |                  90 |          100 |             0.1 |                 0.1 |
| specificity |                  40 |          100 |             0.6 |                 0.2 |
| honesty     |                  10 |           50 |             1.8 |                 0.6 |
| overall     |                  40 |           80 |             0.9 |                -0.3 |
